# SBP verification

**SBP = spider bead pull.** Start with a full bead-pull grid, retain a few transverse lines and axial planes, fit a complex Gaussian field, and compare the reconstructed transverse integrals with the full-grid calculation.

All calculations live in `src/`. This notebook follows the numerical routines from `spider-bead-pull-ai/src/adaptive_bead_pull.py`.

The original **reference-assisted** mode uses the full-grid field to align line-wise square-root signs and complex amplitudes. It tests reduced sampling with that assistance; it is not an independent reconstruction from line data alone. Set `REFERENCE_ALIGNMENT=False` to disable the alignment.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from src import sbp_verification as sbp
from src import verification_plots as plots

## 1. Choose the full-grid measurements

Run Jupyter from the repository root. Put the `.hdf5` grids in `data/full_bead_pull/`, or change `DATA_DIR`. Each file must contain `s11_map` with axes `(x, y, z, frequency)` and JSON attributes `vna_config` and `measurement_config`. Stored positions are in millimetres and are converted to metres.

The initial frequency subset is a short run around 19.5 GHz for the original 18–24 GHz grids. Set `FREQUENCY_INDICES=None` for the complete spectrum. Time gating always runs on the complete acquired spectrum before selecting frequencies.

In [ ]:
DATA_DIR = Path("data/full_bead_pull")
OUTPUT_DIR = Path("outputs/sbp_verification")
LINE_COUNT = 3
Z_SLICE_COUNT = 1
GAP_COUNT = 4
FREQUENCY_INDICES = np.arange(230, 271, 5)
REFERENCE_ALIGNMENT = True
TIME_GATING = True
N_WORKERS = 1

gaps = sbp.discover_gaps(DATA_DIR)
for gap in gaps:
    print(f"Gap {gap.rank}: {gap.name}, acquisition z = {gap.z_min_mm:g}–{gap.z_max_mm:g} mm")

## 2. Select planes and run the reconstruction

Lines use $y=ax+b$ in metres. For one line the central line is used; for two, the two middle lines; for three/four, the first three/four original lines. Points within 3 mm in **y** of each line are retained.

Plane selection follows the old notebook. Ordered by acquisition z, the gap starts correspond to smallest/largest/smallest/fifth-smallest **absolute z**; absolute z runs opposite to acquisition z. The selected indices below make this convention visible.

Processing: time gate → complex affine edge-background subtraction → spatially continuous square root of the reflection change → optional reference alignment → bead-response conversion → complex Gaussian fit. The fit uses the fundamental Gaussian mode with phase curvature. The finite-bead field estimate is $E_{\mathrm{conv}}$; no longitudinal Mie deconvolution is performed here.

In [ ]:
for gap in gaps[:GAP_COUNT]:
    indices = sbp.select_gap_z_ixs(gap, Z_SLICE_COUNT)
    print(f"Gap {gap.rank}: array indices {indices}, acquisition z [mm] = "
          f"{np.asarray(gap.z_positions_m)[list(indices)] * 1e3}")

result = sbp.verify_scan(
    gaps, line_count=LINE_COUNT, z_slice_count=Z_SLICE_COUNT,
    gap_count=GAP_COUNT, frequency_indices=FREQUENCY_INDICES,
    reference_alignment=REFERENCE_ALIGNMENT, time_gating=TIME_GATING,
    n_workers=N_WORKERS,
)
result["metrics"]

## 3. Compare the transverse fields

White dots mark the retained line samples. The same color scale is used for the full-grid and fitted magnitudes. The residual uses the complex fields and is normalized by the full-grid RMS.

In [ ]:
example = result["example"]
f_index = int(np.argmin(abs(result["frequencies"] - 19.5e9)))
field_figure = plots.plot_fields(example, frequency_index=f_index)
plt.show()

## 4. Compare transverse integrals

$E_T(z,f)=\int E_{\mathrm{conv}}(x,y,z,f)\,dx\,dy$ has units **V m**. The reference uses the original grid sum times cell area. The reduced reconstruction uses the original 50×50 trapezoidal integral over the same rectangular bounds. Both conventions are retained to reproduce the old outputs.

The line Gaussian assumes the unmeasured transverse shape. A single line cannot determine the perpendicular beam centre independently; fitted centres/widths are model assumptions rather than independent measurements. The input-power convention is 1 W and the alumina bead diameter/permittivity are 2.93 mm/9.23.

In [ ]:
integral_figure = plots.plot_integrals(result, frequency_index=f_index)
error_figure = plots.plot_integral_error(result)
plt.show()

## 5. Save the ET data and figures

The NPZ retains the original `ET_results.npz` keys: `ETs_reduced`, `ETs_full`, `z_reduced`, `z_full`, `frequencies`. ET axes are `(z, frequency)`; z is the positive relative coordinate stored in the original grids, not an absolute lab coordinate. Configuration and alignment settings are saved too. These integrals can be passed to the separate longitudinal/boost analysis.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sbp.save_results(result, OUTPUT_DIR / "ET_results.npz")
field_figure.savefig(OUTPUT_DIR / "fields.png", dpi=150)
integral_figure.savefig(OUTPUT_DIR / "transverse_integrals.png", dpi=150)
error_figure.savefig(OUTPUT_DIR / "integral_error.png", dpi=150)

## Other configurations

Change `LINE_COUNT` (1–4), `Z_SLICE_COUNT` (1–3) or `GAP_COUNT` (1–4), then rerun from step 2. The full-grid reference always retains all acquired planes. A small parameter sweep is ordinary Python; no scheduler or cache manager is required.